# LoRA Fine-Tuning

This notebook fine-tunes a small language model using LoRA.
We start with a tiny pretrained model add LoRA adapters and
train on a small instruction dataset. At the end you will see
how the model learns to follow instructions without changing
its original weights.

The base model has 17 million parameters. The LoRA adapters
add only 0.3 million trainable parameters. The original
weights stay frozen. This is the power of LoRA.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
import tiktoken

## The base model

A tiny GPT with the same architecture we built in the guide.
We keep it small so LoRA training finishes in seconds.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=128, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)


def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = attn_weights @ v
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


@dataclass
class GPTConfig:
    vocab_size: int = 50257
    d_model: int = 256
    num_heads: int = 4
    num_layers: int = 4
    max_seq_len: int = 128
    dropout: float = 0.1
    embd_dropout: float = 0.1


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embd_dropout = nn.Dropout(config.embd_dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(config.d_model, config.num_heads, config.dropout)
            for _ in range(config.num_layers)
        ])
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if hasattr(module, 'bias') and module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape
        x = self.token_embedding(input_ids)
        x = self.embd_dropout(x)
        mask = create_causal_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            logits_flat = logits.contiguous().view(-1, self.config.vocab_size)
            targets_flat = targets.contiguous().view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            if input_ids.shape[1] > self.config.max_seq_len:
                input_ids = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self.forward(input_ids)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids


config = GPTConfig()
model = GPT(config)
model.eval()

print(f"Base model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Generate before fine-tuning

The base model knows language but has never been fine-tuned to
follow instructions. Let us see what it produces.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

def chat(model, prompt, max_tokens=30):
    full_prompt = f"User: {prompt}\nAssistant:"
    ids = torch.tensor([tokenizer.encode(full_prompt)], dtype=torch.long)
    output = model.generate(ids, max_new_tokens=max_tokens)
    return tokenizer.decode(output[0].tolist())

prompts = [
    "What is the capital of France?",
    "Write a short poem about a cat.",
    "Explain gravity in simple terms.",
]

print("=== BEFORE FINE-TUNING ===")
for p in prompts:
    print(f"\nPrompt: {p}")
    print(f"Output: {chat(model, p)[:200]}")

## Add LoRA adapters

We freeze the base model and add LoRA to the query and value
projections in every attention layer. Only the LoRA weights
will be trained.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, original_linear, rank=8, alpha=16):
        super().__init__()
        self.original = original_linear
        self.rank = rank
        self.alpha = alpha

        in_features = original_linear.in_features
        out_features = original_linear.out_features

        self.lora_a = nn.Linear(in_features, rank, bias=False)
        self.lora_b = nn.Linear(rank, out_features, bias=False)

        nn.init.normal_(self.lora_a.weight, std=0.02)
        nn.init.zeros_(self.lora_b.weight)

        for param in original_linear.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.original(x) + self.lora_b(self.lora_a(x)) * (self.alpha / self.rank)


def add_lora_to_model(model, rank=8, alpha=16):
    for param in model.parameters():
        param.requires_grad = False

    lora_params = []
    for layer in model.layers:
        attn = layer.attention
        original_qkv = attn.qkv_proj

        in_dim = original_qkv.in_features
        out_dim = original_qkv.out_features

        lora_qkv = LoRALinear(original_qkv, rank=rank, alpha=alpha)
        attn.qkv_proj = lora_qkv

        for param in lora_qkv.lora_a.parameters():
            lora_params.append(param)
        for param in lora_qkv.lora_b.parameters():
            lora_params.append(param)

    return lora_params


lora_params = add_lora_to_model(model, rank=8, alpha=16)

base_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
lora_count = sum(p.numel() for p in lora_params)

print(f"Base model (frozen): {base_params:,} parameters")
print(f"LoRA adapters (trainable): {lora_count:,} parameters")
print(f"LoRA adds only {100 * lora_count / base_params:.2f}% more parameters")

## Create a small instruction dataset

A few examples to teach the model to respond like an assistant.
The model will learn the pattern: User asks something. Assistant
gives a helpful response.

In [ ]:
training_data = [
    ("What is the capital of France?", "The capital of France is Paris."),
    ("What is 2 + 2?", "2 + 2 equals 4."),
    ("Who wrote Romeo and Juliet?", "William Shakespeare wrote Romeo and Juliet."),
    ("What color is the sky?", "The sky is blue on a clear day."),
    ("How many legs does a cat have?", "A cat has four legs."),
    ("What is the largest planet?", "Jupiter is the largest planet in our solar system."),
    ("Name a primary color.", "Red is a primary color."),
    ("What sound does a dog make?", "A dog barks."),
    ("What is water made of?", "Water is made of hydrogen and oxygen."),
    ("How many days in a week?", "There are seven days in a week."),
] * 5

print(f"Training examples: {len(training_data)}")

# Show a few
for q, a in training_data[:3]:
    print(f"  Q: {q}")
    print(f"  A: {a}")
    print()

## Train the LoRA adapters

Only the LoRA weights are updated. The base model stays frozen.
This trains in seconds even on CPU.

In [ ]:
optimizer = torch.optim.AdamW(lora_params, lr=3e-4)

tokenizer = tiktoken.get_encoding("gpt2")

print(f"Training for 200 steps...")
start = time.time()

for step in range(200):
    total_loss = 0
    for question, answer in training_data:
        text = f"User: {question}\nAssistant: {answer}"
        ids = tokenizer.encode(text)
        ids = torch.tensor([ids], dtype=torch.long)

        input_ids = ids[:, :-1]
        target_ids = ids[:, 1:]

        if input_ids.shape[1] > config.max_seq_len:
            continue

        _, loss = model(input_ids, target_ids)
        total_loss += loss.item()
        loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if step % 50 == 0 or step == 199:
        elapsed = time.time() - start
        print(f"  Step {step:>3d}: loss={total_loss / len(training_data):.4f} ({elapsed:.0f}s)")

print(f"\nDone in {time.time() - start:.1f}s")

## Generate after fine-tuning

The same prompts. The same model. Now with LoRA adapters
trained to respond like an assistant.

In [ ]:
model.eval()

print("=== AFTER FINE-TUNING ===")
for p in prompts:
    print(f"\nPrompt: {p}")
    print(f"Output: {chat(model, p)[:200]}")

## What changed

The base model weights are exactly the same as before training.
Only the LoRA adapters were updated. You can prove this by
removing the adapters and generating again. The model will
revert to its original behavior.

The adapter file is tiny. You could train adapters for
summarization and translation and code generation and swap
them in and out without reloading the base model. One model.
Many skills. This is LoRA.

## Try your own prompts

In [ ]:
your_prompts = [
    "What is the tallest mountain?",
    "Explain what a computer does.",
]

for p in your_prompts:
    print(f"Prompt: {p}")
    print(f"Output: {chat(model, p)[:200]}")
    print()